# COVID-19 Trends Dashboard — Interactive Analysis

An exploratory walkthrough of global COVID-19 patterns using the Our World in Data dataset.

**Questions explored:**
1. How did case rates evolve globally? Can we identify distinct waves?
2. Which countries were hit hardest, per-capita?
3. How did vaccination coverage vary by wealth and geography?
4. Does the reported COVID death count tell the full story? (Spoiler: no.)
5. How did government response (stringency) track with case loads?

---

## Setup

In [ ]:
import sys
from pathlib import Path

# Make the src/ package importable from the notebook
sys.path.insert(0, str(Path('..').resolve() / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_data, split_countries_and_aggregates
from analysis import add_case_fatality_rate, rolling_average, top_n_by_metric, summarize_country
from visualizations import apply_style

apply_style()
pd.set_option('display.max_columns', 80)

## 1. Load and explore the data

The dataset mixes country rows with aggregate rows (continents, income groups, "World"). Aggregate rows have no `continent` value — we split them out so we never accidentally double-count.

In [ ]:
df = load_data()
countries, aggregates = split_countries_and_aggregates(df)

print(f"Total rows:   {len(df):,}")
print(f"Countries:    {countries['location'].nunique()}")
print(f"Aggregates:   {sorted(aggregates['location'].unique())}")
print(f"Date range:   {df['date'].min().date()} → {df['date'].max().date()}")

## 2. Global headline numbers

In [ ]:
world = aggregates[aggregates['location'] == 'World'].sort_values('date')

def last_valid(col):
    v = world[col].dropna()
    return v.iloc[-1] if not v.empty else None

summary = pd.Series({
    'Total cases': f"{last_valid('total_cases'):,.0f}",
    'Total deaths': f"{last_valid('total_deaths'):,.0f}",
    'Cases per million': f"{last_valid('total_cases_per_million'):,.0f}",
    'Deaths per million': f"{last_valid('total_deaths_per_million'):,.0f}",
    '% fully vaccinated': f"{last_valid('people_fully_vaccinated_per_hundred'):.1f}%",
})
summary.to_frame('value')

## 3. Identifying global waves

The smoothed global case curve shows several distinct waves. The biggest by far is the Omicron wave in early 2022, which peaked at over 3 million cases per day globally. Note the spike in late 2022 — that's largely China's reporting jump after dropping zero-COVID policies.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
ts = world.set_index('date')['new_cases_smoothed']
ax.fill_between(ts.index, ts.values, alpha=0.25)
ax.plot(ts.index, ts.values, linewidth=1.8)
ax.set_title('Global daily new cases (7-day rolling avg)')
ax.set_ylabel('Cases per day')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
plt.show()

## 4. Per-capita rankings: who got hit hardest?

Looking at raw case counts is misleading because of population differences. Per-million metrics reveal that small, well-tested European countries (and South Korea) actually report the highest per-capita case rates. This says more about testing infrastructure than infection prevalence.

In [ ]:
top = top_n_by_metric(countries, 'total_cases_per_million', n=15, min_population=1_000_000)
rankings = (countries[countries['location'].isin(top)]
            .dropna(subset=['total_cases_per_million'])
            .groupby('location', observed=True)
            .tail(1)
            .sort_values('total_cases_per_million', ascending=False)
            [['location', 'total_cases_per_million', 'total_deaths_per_million']]
            .reset_index(drop=True))
rankings

## 5. Vaccination equity — wealth vs coverage

Plotting GDP per capita against vaccination rate reveals a striking divide. High-income countries clustered around 70–90% coverage, while many low-income countries (especially in sub-Saharan Africa) never crossed 30%.

In [ ]:
valid = countries.dropna(subset=['people_fully_vaccinated_per_hundred', 'gdp_per_capita', 'continent'])
idx = valid.groupby('location', observed=True)['date'].idxmax()
latest = valid.loc[idx]

# Group by income brackets
latest['income_bracket'] = pd.cut(
    latest['gdp_per_capita'],
    bins=[0, 5000, 15000, 35000, 1e6],
    labels=['Low (<$5k)', 'Lower-mid ($5–15k)', 'Upper-mid ($15–35k)', 'High (>$35k)']
)

summary = (latest.groupby('income_bracket', observed=True)['people_fully_vaccinated_per_hundred']
                .agg(['mean', 'median', 'count'])
                .round(1))
summary.columns = ['Mean %', 'Median %', 'Countries']
summary

## 6. The mortality undercount — excess mortality vs reported deaths

This is one of the most important comparisons in the dataset. Excess mortality (deaths above the historical baseline) captures the pandemic's *true* toll, including uncounted COVID deaths and indirect deaths from overwhelmed health systems. For many countries — Russia, Bulgaria, Serbia, South Africa — excess mortality is **2–3× higher** than the reported COVID death count, suggesting massive undercounting.

In [ ]:
def latest_valid_value(group, col):
    v = group[col].dropna()
    return v.iloc[-1] if not v.empty else np.nan

grouped = countries.groupby('location', observed=True)
comp = pd.DataFrame({
    'excess_mortality_per_M': grouped.apply(lambda g: latest_valid_value(g, 'excess_mortality_cumulative_per_million')),
    'reported_deaths_per_M': grouped.apply(lambda g: latest_valid_value(g, 'total_deaths_per_million')),
    'population': grouped['population'].max(),
}).reset_index().dropna(subset=['excess_mortality_per_M', 'reported_deaths_per_M'])

comp = comp.query('population >= 3_000_000').copy()
comp['ratio'] = comp['excess_mortality_per_M'] / comp['reported_deaths_per_M']
comp.sort_values('ratio', ascending=False).head(15)[['location', 'excess_mortality_per_M', 'reported_deaths_per_M', 'ratio']]

## 7. Country deep-dive

Pick any country to get a quick summary.

In [ ]:
for country in ['United States', 'Brazil', 'India', 'Japan', 'South Africa']:
    s = summarize_country(countries, country)
    print(f"\n{s['country']}")
    print(f"  Population:      {s['population']:>15,.0f}")
    print(f"  Total cases:     {s['total_cases']:>15,.0f}")
    print(f"  Total deaths:    {s['total_deaths']:>15,.0f}")
    print(f"  Deaths/M:        {s['deaths_per_million']:>15,.0f}")
    print(f"  Fully vax:       {s['pct_fully_vaccinated']:>14.1f}%")
    print(f"  Peak day:        {s['peak_daily_cases_date']} ({s['peak_daily_cases']:,.0f} cases)")

## Next steps

Run the full dashboard script to generate the complete set of charts:

```bash
python src/dashboard.py
```

This produces 10 publication-ready PNGs in `outputs/`.